In [1]:
import sqlite3

print("SQLite version:", sqlite3.sqlite_version)
print("SQLite is working!")

SQLite version: 3.50.4
SQLite is working!


In [2]:
import sqlite3
import pandas as pd
import os

In [3]:
df = pd.read_csv("../data/processed/telco_churn_clean.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
os.makedirs("../database", exist_ok=True)

conn = sqlite3.connect("../database/retainiq.db")

print("Database connected successfully!")

Database connected successfully!


In [5]:
df.to_sql(
    "customers",
    conn,
    if_exists="replace",
    index=False
)

print("Customer data inserted successfully!")

Customer data inserted successfully!


In [6]:
cursor = conn.cursor()

cursor.execute("SELECT COUNT(*) FROM customers")

print("Total records:", cursor.fetchone()[0])

Total records: 7043


In [7]:
query = """
SELECT COUNT(*) AS total_customers
FROM customers;
"""

result = pd.read_sql_query(query, conn)
result

,total_customers
0,7043


In [8]:
query = """
SELECT Churn, COUNT(*) AS customer_count
FROM customers
GROUP BY Churn;
"""

pd.read_sql_query(query, conn)

,Churn,customer_count
0,No,5174
1,Yes,1869


In [9]:
query = """
SELECT
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*), 2
    ) AS churn_rate
FROM customers;
"""

pd.read_sql_query(query, conn)

,total_customers,churned_customers,churn_rate
0,7043,1869,26.54


In [10]:
query = """
SELECT
    Contract,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*), 2
    ) AS churn_rate
FROM customers
GROUP BY Contract
ORDER BY churn_rate DESC;
"""

pd.read_sql_query(query, conn)

,Contract,total_customers,churned_customers,churn_rate
0,Month-to-month,3875,1655,42.71
1,One year,1473,166,11.27
2,Two year,1695,48,2.83


In [11]:
query = """
SELECT
    CASE
        WHEN tenure <= 12 THEN 'New Customers'
        WHEN tenure <= 36 THEN 'Medium-Term Customers'
        ELSE 'Long-Term Customers'
    END AS tenure_group,

    COUNT(*) AS total_customers,

    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        AS churned_customers,

    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*), 2
    ) AS churn_rate

FROM customers
GROUP BY tenure_group
ORDER BY churn_rate DESC;
"""

pd.read_sql_query(query, conn)

,tenure_group,total_customers,churned_customers,churn_rate
0,New Customers,2186,1037,47.44
1,Medium-Term Customers,1856,474,25.54
2,Long-Term Customers,3001,358,11.93


In [12]:
query = """
SELECT
    PaymentMethod,
    COUNT(*) AS total_customers,

    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        AS churned_customers,

    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*), 2
    ) AS churn_rate

FROM customers
GROUP BY PaymentMethod
ORDER BY churn_rate DESC;
"""

pd.read_sql_query(query, conn)

,PaymentMethod,total_customers,churned_customers,churn_rate
0,Electronic check,2365,1071,45.29
1,Mailed check,1612,308,19.11
2,Bank transfer (automatic),1544,258,16.71
3,Credit card (automatic),1522,232,15.24


In [13]:
query = """
SELECT
    InternetService,
    COUNT(*) AS total_customers,

    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        AS churned_customers,

    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*), 2
    ) AS churn_rate

FROM customers
GROUP BY InternetService
ORDER BY churn_rate DESC;
"""

pd.read_sql_query(query, conn)

,InternetService,total_customers,churned_customers,churn_rate
0,Fiber optic,3096,1297,41.89
1,DSL,2421,459,18.96
2,No,1526,113,7.40


In [14]:
query = """
SELECT
    customerID,
    Contract,
    tenure,
    MonthlyCharges,
    TotalCharges,
    Churn

FROM customers

WHERE Churn = 'Yes'
AND MonthlyCharges > (
    SELECT AVG(MonthlyCharges)
    FROM customers
)

ORDER BY MonthlyCharges DESC
LIMIT 15;
"""

pd.read_sql_query(query, conn)

,customerID,Contract,tenure,MonthlyCharges,TotalCharges,Churn
0,8199-ZLLSA,One year,67,118.35,7804.15,Yes
1,2889-FPWRM,One year,72,117.80,8684.80,Yes
2,2302-ANTDP,Month-to-month,48,117.45,5438.90,Yes
3,9053-JZFKV,Two year,67,116.20,7752.30,Yes
4,1444-VVSGW,One year,70,115.65,7968.85,Yes
5,0201-OAMXR,One year,70,115.55,8127.60,Yes
6,4361-BKAXE,Month-to-month,41,114.50,4527.45,Yes
7,1555-DJEQW,Two year,70,114.20,7723.90,Yes
8,9158-VCTQB,Month-to-month,41,113.60,4594.95,Yes
9,7279-BUYWN,Month-to-month,41,113.20,4689.50,Yes


In [15]:
query = """
WITH customer_risk AS (
    SELECT
        customerID,
        Contract,
        tenure,
        InternetService,
        PaymentMethod,
        MonthlyCharges,
        Churn,

        (
            CASE WHEN Contract = 'Month-to-month' THEN 1 ELSE 0 END +
            CASE WHEN tenure <= 12 THEN 1 ELSE 0 END +
            CASE WHEN InternetService = 'Fiber optic' THEN 1 ELSE 0 END +
            CASE WHEN PaymentMethod = 'Electronic check' THEN 1 ELSE 0 END +
            CASE WHEN MonthlyCharges > (
                SELECT AVG(MonthlyCharges) FROM customers
            ) THEN 1 ELSE 0 END
        ) AS risk_score

    FROM customers
)

SELECT
    CASE
        WHEN risk_score >= 3 THEN 'High Risk'
        WHEN risk_score = 2 THEN 'Medium Risk'
        ELSE 'Low Risk'
    END AS risk_segment,

    COUNT(*) AS total_customers,

    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        AS churned_customers,

    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*), 2
    ) AS churn_rate

FROM customer_risk
GROUP BY risk_segment
ORDER BY
    CASE risk_segment
        WHEN 'High Risk' THEN 1
        WHEN 'Medium Risk' THEN 2
        ELSE 3
    END;
"""

risk_results = pd.read_sql_query(query, conn)
risk_results

,risk_segment,total_customers,churned_customers,churn_rate
0,High Risk,2812,1388,49.36
1,Medium Risk,1772,354,19.98
2,Low Risk,2459,127,5.16


In [16]:
query = """
CREATE VIEW IF NOT EXISTS customer_risk_view AS

SELECT
    customerID,
    Contract,
    tenure,
    InternetService,
    PaymentMethod,
    MonthlyCharges,
    Churn,

    (
        CASE WHEN Contract = 'Month-to-month' THEN 1 ELSE 0 END +
        CASE WHEN tenure <= 12 THEN 1 ELSE 0 END +
        CASE WHEN InternetService = 'Fiber optic' THEN 1 ELSE 0 END +
        CASE WHEN PaymentMethod = 'Electronic check' THEN 1 ELSE 0 END +
        CASE WHEN MonthlyCharges > (
            SELECT AVG(MonthlyCharges) FROM customers
        ) THEN 1 ELSE 0 END
    ) AS risk_score

FROM customers;
"""

cursor.execute(query)
conn.commit()

print("Customer risk view created successfully!")

Customer risk view created successfully!


In [17]:
pd.read_sql_query("""
SELECT
    customerID,
    risk_score,
    Churn
FROM customer_risk_view
ORDER BY risk_score DESC
LIMIT 10;
""", conn)

,customerID,risk_score,Churn
0,9237-HQITU,5,Yes
1,9305-CDSKC,5,Yes
2,6047-YHPVI,5,Yes
3,7760-OYPDY,5,Yes
4,5698-BQJOH,5,Yes
5,5919-TMRGD,5,Yes
6,2424-WVHPL,5,No
7,8637-XJIVR,5,Yes
8,3071-VBYPO,5,No
9,5299-RULOA,5,Yes


In [18]:
query = """
CREATE VIEW customer_risk_dashboard AS

SELECT
    customerID,
    Contract,
    tenure,
    InternetService,
    PaymentMethod,
    MonthlyCharges,
    Churn,
    risk_score,

    CASE
        WHEN risk_score >= 3 THEN 'High Risk'
        WHEN risk_score = 2 THEN 'Medium Risk'
        ELSE 'Low Risk'
    END AS risk_segment

FROM customer_risk_view;
"""

cursor.execute("DROP VIEW IF EXISTS customer_risk_dashboard")
cursor.execute(query)
conn.commit()

print("Customer risk dashboard view created!")

Customer risk dashboard view created!


In [19]:
pd.read_sql_query("""
SELECT *
FROM customer_risk_dashboard
ORDER BY risk_score DESC
LIMIT 10;
""", conn)

,customerID,Contract,tenure,InternetService,PaymentMethod,MonthlyCharges,Churn,risk_score,risk_segment
0,9237-HQITU,Month-to-month,2,Fiber optic,Electronic check,70.70,Yes,5,High Risk
1,9305-CDSKC,Month-to-month,8,Fiber optic,Electronic check,99.65,Yes,5,High Risk
2,6047-YHPVI,Month-to-month,5,Fiber optic,Electronic check,69.70,Yes,5,High Risk
3,7760-OYPDY,Month-to-month,2,Fiber optic,Electronic check,80.65,Yes,5,High Risk
4,5698-BQJOH,Month-to-month,9,Fiber optic,Electronic check,94.40,Yes,5,High Risk
5,5919-TMRGD,Month-to-month,1,Fiber optic,Electronic check,79.35,Yes,5,High Risk
6,2424-WVHPL,Month-to-month,1,Fiber optic,Electronic check,74.70,No,5,High Risk
7,8637-XJIVR,Month-to-month,12,Fiber optic,Electronic check,78.95,Yes,5,High Risk
8,3071-VBYPO,Month-to-month,3,Fiber optic,Electronic check,89.85,No,5,High Risk
9,5299-RULOA,Month-to-month,10,Fiber optic,Electronic check,100.25,Yes,5,High Risk


In [20]:
# Export cleaned customer data
df.to_csv(
    "../data/processed/customers_for_powerbi.csv",
    index=False
)

# Export customer risk segmentation
risk_df = pd.read_sql_query(
    "SELECT * FROM customer_risk_dashboard",
    conn
)

risk_df.to_csv(
    "../data/processed/customer_risk_segments.csv",
    index=False
)

print("Power BI datasets exported successfully!")

Power BI datasets exported successfully!
